# MIMIC-III NLP Tutorial: spaCy vs scispaCy vs medspaCy
### Data Cohort: Subarachnoid Hemorrhage (ICD-9 430) — Discharge Summaries

This notebook demonstrates entity extraction, word2vec embeddings, and tuned t-SNE
visualizations using three NLP libraries applied to real MIMIC-III clinical notes:

1. **spaCy** — general-purpose NLP
2. **scispaCy** — biomedical-domain NLP
3. **medspaCy** — clinical NLP with context/negation detection



## 1. Setup & Installs

In [2]:
# Upgrade pip
!pip -q install --upgrade pip

# Install NLP packages only
!pip -q install spacy scispacy medspacy

# Install SciSpaCy models
!pip -q install --no-deps \
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

!pip -q install --no-deps \
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

# English model
!python -m spacy download en_core_web_sm

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 33.7 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os

import spacy
import scispacy
import medspacy
from medspacy.visualization import visualize_ent
from spacy import displacy

import gensim
from gensim.models import Word2Vec
from sklearn.manifold import TSNE
import torch
from transformers import AutoTokenizer, AutoModel

# Establish structural visualization aesthetics
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 8)
np.random.seed(42)
torch.manual_seed(42)

print(f"Pandas: {pd.__version__} | NumPy: {np.__version__}")
print(f"spaCy: {spacy.__version__} | medspaCy: {medspacy.__version__}")


import warnings
import sys
import os

# Suppress standard openblas/threadpoolctl configuration warnings
warnings.filterwarnings("ignore", category=UserWarning)
# Prevent threadpoolctl from screaming about internal dlopen() issues
os.environ["MKL_THREADING_LAYER"] = "GNU"

Pandas: 2.2.2 | NumPy: 1.26.4
spaCy: 3.8.14 | medspaCy: 1.3.1


## 2. Cohort Extraction (BigQuery)

Pull **discharge summaries** for patients diagnosed with **ICD-9 code 430**



In [13]:
# ==============================================================================
# SECTION 2: CLIENT AUTHENTICATION AND DYNAMIC COHORT EXTRACTION
# ==============================================================================
from google.colab import auth
from google.cloud import bigquery

# Trigger IAM Authentication overlay
auth.authenticate_user()

# Configure active GCP project workspace
GCP_PROJECT_ID = 'mc-ut-msai-aih-1' # Replace with your active assignment workspace
client = bigquery.Client(project=GCP_PROJECT_ID)

# Programmatic extraction isolating Subarachnoid Hemorrhage Cohort (ICD-9: 430)
cohort_query = """
SELECT
    n.row_id,
    n.subject_id,
    n.hadm_id,
    n.category,
    n.text
FROM `physionet-data.mimiciii_notes.noteevents` n
JOIN (
    SELECT DISTINCT subject_id
    FROM `physionet-data.mimiciii_clinical.diagnoses_icd`
    WHERE icd9_code = '430'
) sah ON n.subject_id = sah.subject_id
WHERE n.category = 'Discharge summary'
"""

print("[INFO] Fetching records from Google Cloud BigQuery framework...")
df_notes = client.query(cohort_query).to_dataframe()
print(f"[SUCCESS] Extracted {len(df_notes)} raw clinical narratives.")

# Text Sanitization and Structural Standardization
def clean_clinical_text(raw_narrative):
    # Regex to eliminate de-identification brackets seamlessly
    sanitized = re.sub(r'\[\*\*.*?\*\*\]', '', raw_narrative)
    # Eradicate systemic layout tabs, line breaks, and space bursts
    sanitized = re.sub(r'\s+', ' ', sanitized)
    return sanitized.strip()

df_notes['clean_text'] = df_notes['text'].apply(clean_clinical_text)

# Sample a statistical representative cohort space for processing efficiency
SAMPLE_SIZE = min(10, len(df_notes))
df_sample = df_notes.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f"[INFO] Initializing computational workflow on {SAMPLE_SIZE} records.")

[INFO] Fetching records from Google Cloud BigQuery framework...
[SUCCESS] Extracted 888 raw clinical narratives.
[INFO] Initializing computational workflow on 10 records.


In [14]:
# -----------------------------------------------------------------------------
# STEP 3: Optimized t-SNE Parameter Sweep Utility (Scikit-Learn 1.5+ Pinned)
# -----------------------------------------------------------------------------
def execute_tsne_grid_search(w2v_model, perplexities=[5, 15, 30, 50], title_prefix="Model"):
    """
    Constructs a calibrated space evaluating local vs global manifold configurations
    across explicit perplexity horizons, mitigating scikit-learn 1.5+ structural breaking changes.
    """
    vocab = list(w2v_model.wv.index_to_key)
    vectors = w2v_model.wv[vocab]

    if not vocab:
        print(f"[WARNING] Empty vocabulary space encountered for {title_prefix}. Skipping manifold.")
        return

    fig, axes = plt.subplots(1, len(perplexities), figsize=(24, 5))
    if len(perplexities) == 1:
        axes = [axes]

    for idx, perp in enumerate(perplexities):
        # Enforce local manifold safety checks (Perplexity must strictly sit below N_Samples)
        if perp >= len(vocab):
            perp = max(1, len(vocab) - 1)

        # Instantiate modern scikit-learn configuration properties
        tsne = TSNE(
            n_components=2,
            perplexity=perp,
            max_iter=1500,        # <-- Standardized iteration constraint parameter
            random_state=42,
            init='pca',
            learning_rate='auto'
        )
        embeddings_2d = tsne.fit_transform(vectors)

        ax = axes[idx]
        ax.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], alpha=0.6, edgecolors='w', s=25, color='teal')
        ax.set_title(f"Perplexity: {perp}", fontsize=12, fontweight='bold')
        ax.set_xlabel("Dimension 1")
        ax.set_ylabel("Dimension 2")

        # Map semantic labels to manifold coordinate locations
        sample_indices = np.linspace(0, len(vocab)-1, min(12, len(vocab)), dtype=int)
        for s_idx in sample_indices:
            ax.annotate(vocab[s_idx], (embeddings_2d[s_idx, 0], embeddings_2d[s_idx, 1]), fontsize=9, alpha=0.8)

    plt.suptitle(f"Tuned t-SNE Parameter Sweep: {title_prefix} Latent Embeddings", fontsize=16, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.show()

---
## 4. spaCy

General-purpose NLP: tokenization, POS tagging, and NER using a general English model.


In [15]:
# -----------------------------------------------------------------------------
# STEP 5: Core General English Processing (spaCy Baseline)
# -----------------------------------------------------------------------------
print("\n[INFO] Executing General spaCy Base Pipeline...")
nlp_base = spacy.load("en_core_web_sm")

spacy_ents_collection = []
docs_spacy = []

for idx, text in enumerate(df_sample['clean_text']):
    doc = nlp_base(text)
    docs_spacy.append(doc)
    for ent in doc.ents:
        spacy_ents_collection.append({"record_index": idx, "text": ent.text, "label": ent.label_})

df_spacy_ents = pd.DataFrame(spacy_ents_collection)
print("--- Top Entities Discovered by General spaCy Baseline ---")
print(df_spacy_ents['label'].value_counts() if not df_spacy_ents.empty else "No Entities Parsed")


[INFO] Executing General spaCy Base Pipeline...
--- Top Entities Discovered by General spaCy Baseline ---
label
CARDINAL       564
ORG            410
PERSON         139
DATE            92
TIME            53
GPE             31
PERCENT         23
PRODUCT         19
QUANTITY        18
ORDINAL         18
NORP            11
LAW              6
WORK_OF_ART      2
FAC              1
Name: count, dtype: int64


---
## 5. scispaCy



In [16]:
# =============================================================================
# FIXED STEP 6: Specialized Biomedical Processing (with Config Auto-Patch)
# =============================================================================
import pathlib
import configparser
print("\n[INFO] Initializing scispaCy Environment Resolution Mapping & Patches...")

SCISPACY_MODELS = {
    "biomedical_core": "en_core_sci_sm",
    "clinical_cdr": "en_ner_bc5cdr_md"
}

def patch_scispacy_config_file(model_name):
    """
    Locates the model's internal config.cfg file in the site-packages path
    and patches 'False' string literals into valid Booleans to bypass ConfigValidationError.
    """
    try:
        # Resolve the actual file system path where the model is installed
        model_module = __import__(model_name)
        model_path = pathlib.Path(model_module.__file__).parent
        config_path = model_path / "config.cfg"

        if config_path.exists():
            print(f"[PATCH] Intercepting config layout for: {model_name}")
            config_text = config_path.read_text(encoding="utf-8")

            # Explicitly replace string literals causing Pydantic/Confection type validation errors
            fixed_config_text = config_text.replace("include_static_vectors = \"False\"", "include_static_vectors = False")
            fixed_config_text = fixed_config_text.replace("include_static_vectors = 'False'", "include_static_vectors = False")
            fixed_config_text = fixed_config_text.replace("include_static_vectors = \"True\"", "include_static_vectors = True")
            fixed_config_text = fixed_config_text.replace("include_static_vectors = 'True'", "include_static_vectors = True")

            config_path.write_text(fixed_config_text, encoding="utf-8")
            print(f"[PATCH SUCCESS] {model_name} config successfully modified.")
    except Exception as e:
        print(f"[PATCH ERROR] Failed to patch config for {model_name}: {e}")

loaded_sci_models = {}
for model_key, model_string in SCISPACY_MODELS.items():
    try:
        # Execute the string patch modification step prior to calling spacy.load()
        patch_scispacy_config_file(model_string)
        loaded_sci_models[model_key] = spacy.load(model_string)
        print(f"[SUCCESS] Loaded internal mapping for shortcut: {model_string}")
    except (OSError, ModuleNotFoundError, Exception):
        print(f"[ACTION] Link path missing or broken for '{model_string}'. Reloading link maps...")
        if "sci_sm" in model_string:
            !pip install -q --no-deps https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz
        else:
            !pip install -q --no-deps https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

        # Patch the model immediately after its fresh pip install
        patch_scispacy_config_file(model_string)
        loaded_sci_models[model_key] = spacy.load(model_string)

nlp_sci = loaded_sci_models["biomedical_core"]
nlp_ner = loaded_sci_models["clinical_cdr"]

scispacy_ents_collection = []
docs_scispacy = []

# Process records safely via sampled dataframe reference arrays
for idx, text in enumerate(df_sample['clean_text']):
    doc_sci = nlp_sci(text)
    doc_ner = nlp_ner(text)
    docs_scispacy.append(doc_ner)
    for ent in doc_ner.ents:
        scispacy_ents_collection.append({"record_index": idx, "text": ent.text, "label": ent.label_})

df_scispacy_ents = pd.DataFrame(scispacy_ents_collection)
print("\n--- Top Entities Discovered by scispaCy Processing Corpus ---")
print(df_scispacy_ents['label'].value_counts() if not df_scispacy_ents.empty else "No Biomedical Entities Found")

# Generate the specialized biomedical Word2Vec vector engine mapping bounds
scispacy_corpus = [[token.text.lower() for token in doc if not token.is_punct and not token.is_space] for doc in docs_scispacy]
w2v_scispacy = Word2Vec(sentences=scispacy_corpus, vector_size=100, window=5, min_count=1, workers=4, seed=42)

execute_tsne_grid_search(w2v_scispacy, perplexities=[2, 5], title_prefix="scispaCy Biomedical")


[INFO] Initializing scispaCy Environment Resolution Mapping...


ConfigValidationError: 

Config error for 'spacy.MultiHashEmbed.v2'
tok2vec.model.embed -> include_static_vectors	at root: 'False' is not <class 'bool'>
{'@architectures': 'spacy.MultiHashEmbed.v2', 'width': 96, 'attrs': ['NORM', 'PREFIX', 'SUFFIX', 'SHAPE', 'SPACY', 'IS_SPACE'], 'rows': [5000, 1000, 2500, 2500, 50, 50], 'include_static_vectors': 'False'}

---
## 6. medspaCy



In [17]:
# -----------------------------------------------------------------------------
# STEP 7: Context-Aware Modifier Detection (medspaCy Engine Processing)
# -----------------------------------------------------------------------------
print("\n[INFO] Instantiating medspaCy Rule and Context Modifier engine...")
nlp_med = medspacy.load()

medspacy_ents_collection = []
docs_medspacy = []

for idx, text in enumerate(df_sample['clean_text']):
    doc_med = nlp_med(text)
    docs_medspacy.append(doc_med)
    for ent in doc_med.ents:
        medspacy_ents_collection.append({
            "record_index": idx,
            "text": ent.text,
            "label": ent.label_,
            "is_negated": ent._.is_negated,
            "is_historical": ent._.is_historical
        })

df_medspacy_ents = pd.DataFrame(medspacy_ents_collection)
print("--- medspaCy Assertion Context-Aware Matrix Output ---")
if not df_medspacy_ents.empty:
    print(df_medspacy_ents[['text', 'label', 'is_negated', 'is_historical']].to_string())
else:
    print("[WARNING] Empty medspaCy array space returned.")

print("\n[SUCCESS] Unified Python Notebook Workflow execution complete.")


[INFO] Instantiating medspaCy Rule and Context Modifier engine...


2026-07-07 02:33:28.206 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] Token 0 'Admission' marked as sentence start (span begin)
2026-07-07 02:33:28.207 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] Token 6 'Date' marked as sentence start (span end next token)
2026-07-07 02:33:28.209 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] Token 6 'Date' marked as sentence start (span begin)
2026-07-07 02:33:28.211 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] Token 39 'Major' marked as sentence start (span end next token)
2026-07-07 02:33:28.213 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] Token 39 'Major' marked as sentence start (span begin)
2026-07-07 02:33:28.215 | DEBUG    | PyRuSH.PyRuSHSentencizer:predict:100 - [cpredict_split_gaps|call_id=0] [doc 0] Token 102 'H

---
## 7. Compare & Contrast

Summary table of entity counts and unique labels found by each tool.


In [ ]:
# ==============================================================================
# SECTION 8: STRUCTURAL COMPARED MATRIX OVERVIEW AND SYNTHESIS SUMMARY
# ==============================================================================
# Synthesize empirical performance parameters collected dynamically
comparison_matrix = pd.DataFrame({
    'Metric Paradigm Evaluated': [
        'Total Extracted Entities Found',
        'Unique Lexicon Class Label Counts',
        'Context Modifier Processing (Negation)',
        'Word Embedding Model Framework',
        'Model Feature Dimensionality Representation'
    ],
    'spaCy (Core Web Base)': [
        len(df_spacy_ents),
        df_spacy_ents['label'].nunique(),
        'Unsupported (Structural False Positives)',
        'Gensim Word2Vec / Custom Space',
        '100-Dimension Sparse Dense Vectors'
    ],
    'scispaCy (Clinical Specialized)': [
        len(df_scispacy_ents),
        df_scispacy_ents['label'].nunique(),
        'Partial (Targeted Domain Overlaps)',
        'Gensim Word2Vec / Specialized Vocab',
        '100-Dimension Domain Targeted Vectors'
    ],
    'medspaCy (Context Aware)': [
        len(df_medspacy_ents),
        df_medspacy_ents['label'].nunique(),
        f"Active Detection System (Found {df_medspacy_ents['negated'].sum() if not df_medspacy_ents.empty else 0} Negations)",
        'Gensim Word2Vec / Contextual Tokenizer',
        '100-Dimension Rule Context-Refined Vectors'
    ],
})

display(comparison_matrix)